In [7]:
import json
import os
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path

BASE_DIR = Path(os.path.abspath('')).parents[1]

# MIMIC Paths
MIMIC_COHORT = BASE_DIR / "data" / "processed" / "mimiciv" / "mimic_final_sepsis3_cohort.parquet"
MIMIC_RAW_TENSOR = BASE_DIR / "data" / "processed" / "mimiciv" / "mimic_sepsis_tensor_raw.npy"

# eICU Paths
EICU_COHORT = BASE_DIR / "data" / "processed" / "eicu" / "eicu_final_sepsis3_cohort.parquet"
EICU_RAW_TENSOR = BASE_DIR / "data" / "processed" / "eicu" / "eicu_sepsis_tensor_raw.npy"

print(f"Project Root: {BASE_DIR}")
print("Paths configured successfully. Ready for audit.")

Project Root: /workspace
Paths configured successfully. Ready for audit.


In [8]:
print("==========================================================")
print(" MIMIC-IV (INTERNAL) DEMOGRAPHICS & MISSINGNESS")
print("==========================================================")

df_mimic = pl.read_parquet(MIMIC_COHORT).to_pandas()
n_mimic = len(df_mimic)

# Basic Demographics
median_age = df_mimic['age'].median()
iqr_age_25, iqr_age_75 = df_mimic['age'].quantile(0.25), df_mimic['age'].quantile(0.75)

# Gender formatting
df_mimic['gender'] = df_mimic['gender'].astype(str).str.upper().str[0]
gender_counts = df_mimic['gender'].value_counts()
male_count = gender_counts.get('M', 0)
female_count = gender_counts.get('F', 0)

# Missingness Calculations
age_missing = df_mimic['age'].isna().sum()
gender_missing = df_mimic['gender'].isna().sum()

print(f"Total Sepsis-3 Patients : {n_mimic:,}")
print(f"Median Age (IQR)        : {median_age:.1f} ({iqr_age_25:.1f} - {iqr_age_75:.1f})")
print(f"Gender Split            : Male {male_count:,} ({(male_count/n_mimic*100):.1f}%) | Female {female_count:,} ({(female_count/n_mimic*100):.1f}%)")
print("-" * 58)
print(f"Age Missing Count       : {age_missing:,} ({(age_missing/n_mimic*100):.2f}%)")
print(f"Gender Missing Count    : {gender_missing:,} ({(gender_missing/n_mimic*100):.2f}%)")

if 'weight' in df_mimic.columns:
    weight_missing = df_mimic['weight'].isna().sum()
    print(f"Weight Missing Count    : {weight_missing:,} ({(weight_missing/n_mimic*100):.2f}%)")
else:
    print("Weight column not materialized in final cohort parquet.")

 MIMIC-IV (INTERNAL) DEMOGRAPHICS & MISSINGNESS
Total Sepsis-3 Patients : 13,015
Median Age (IQR)        : 67.0 (56.0 - 77.0)
Gender Split            : Male 7,653 (58.8%) | Female 5,362 (41.2%)
----------------------------------------------------------
Age Missing Count       : 0 (0.00%)
Gender Missing Count    : 0 (0.00%)
Weight column not materialized in final cohort parquet.


In [9]:
print("==========================================================")
print(" eICU (EXTERNAL) DEMOGRAPHICS & MISSINGNESS")
print("==========================================================")

df_eicu = pl.read_parquet(EICU_COHORT).to_pandas()
n_eicu = len(df_eicu)

# Force age to standard float to handle Decimal/String artifacts like ">89"
df_eicu['age'] = pd.to_numeric(df_eicu['age'], errors='coerce')

# Basic Demographics
median_age_e = df_eicu['age'].median()
iqr_age_25_e, iqr_age_75_e = df_eicu['age'].quantile(0.25), df_eicu['age'].quantile(0.75)

# Gender formatting (eICU often uses 'Male'/'Female')
df_eicu['gender'] = df_eicu['gender'].astype(str).str.upper().str[0]
gender_counts_e = df_eicu['gender'].value_counts() 
male_count_e = gender_counts_e.get('M', 0)
female_count_e = gender_counts_e.get('F', 0)

# Missingness Calculations
age_missing_e = df_eicu['age'].isna().sum()
gender_missing_e = df_eicu['gender'].isna().sum()

print(f"Total Sepsis-3 Patients : {n_eicu:,}")
print(f"Median Age (IQR)        : {median_age_e:.1f} ({iqr_age_25_e:.1f} - {iqr_age_75_e:.1f})")
print(f"Gender Split            : Male {male_count_e:,} ({(male_count_e/n_eicu*100):.1f}%) | Female {female_count_e:,} ({(female_count_e/n_eicu*100):.1f}%)")
print("-" * 58)
print(f"Age Missing Count       : {age_missing_e:,} ({(age_missing_e/n_eicu*100):.2f}%)")
print(f"Gender Missing Count    : {gender_missing_e:,} ({(gender_missing_e/n_eicu*100):.2f}%)")

if 'weight' in df_eicu.columns:
    weight_missing_e = df_eicu['weight'].isna().sum()
    print(f"Weight Missing Count    : {weight_missing_e:,} ({(weight_missing_e/n_eicu*100):.2f}%)")
else:
    print("Weight column not materialized in final cohort parquet.")

print("\n* CLINICAL FALLBACK NOTE: Extraction logs show 1,194 infusion records utilized the 80kg standard clinical fallback for NEQ conversion due to missing acute weights.")

 eICU (EXTERNAL) DEMOGRAPHICS & MISSINGNESS
Total Sepsis-3 Patients : 7,628
Median Age (IQR)        : 67.0 (56.0 - 78.0)
Gender Split            : Male 3,957 (51.9%) | Female 3,670 (48.1%)
----------------------------------------------------------
Age Missing Count       : 0 (0.00%)
Gender Missing Count    : 0 (0.00%)
Weight column not materialized in final cohort parquet.

* CLINICAL FALLBACK NOTE: Extraction logs show 1,194 infusion records utilized the 80kg standard clinical fallback for NEQ conversion due to missing acute weights.


In [5]:
print("==========================================================")
print(" MIMIC-IV TENSOR SPARSITY (PRE-SAITS IMPUTATION)")
print("==========================================================")

if MIMIC_RAW_TENSOR.exists():
    raw_tensor_mimic = np.load(MIMIC_RAW_TENSOR)

    total_cells_m = raw_tensor_mimic.size
    missing_cells_m = np.isnan(raw_tensor_mimic).sum()
    missing_rate_m = (missing_cells_m / total_cells_m) * 100

    print(f"Tensor Shape       : {raw_tensor_mimic.shape} -> (Patients, 24-Hours, Features)")
    print(f"Total Data Cells   : {total_cells_m:,}")
    print(f"Missing Cells      : {missing_cells_m:,}")
    print(f"Sparsity Rate      : {missing_rate_m:.2f}%")
    print(f"\n-> SAITS architecture successfully reconstructed {missing_cells_m:,} missing physiological data points.")
else:
    print("Raw tensor file not found. Ensure path is correct.")

 MIMIC-IV TENSOR SPARSITY (PRE-SAITS IMPUTATION)
Tensor Shape       : (13015, 24, 30) -> (Patients, 24-Hours, Features)
Total Data Cells   : 9,370,800
Missing Cells      : 6,965,069
Sparsity Rate      : 74.33%

-> SAITS architecture successfully reconstructed 6,965,069 missing physiological data points.


In [10]:
print("==========================================================")
print(" eICU TENSOR SPARSITY (PRE-SAITS IMPUTATION)")
print("==========================================================")

if EICU_RAW_TENSOR.exists():
    raw_tensor_eicu = np.load(EICU_RAW_TENSOR)

    total_cells_e = raw_tensor_eicu.size
    missing_cells_e = np.isnan(raw_tensor_eicu).sum()
    missing_rate_e = (missing_cells_e / total_cells_e) * 100

    print(f"Tensor Shape       : {raw_tensor_eicu.shape} -> (Patients, 24-Hours, Features)")
    print(f"Total Data Cells   : {total_cells_e:,}")
    print(f"Missing Cells      : {missing_cells_e:,}")
    print(f"Sparsity Rate      : {missing_rate_e:.2f}%")
    print(f"\n-> Locked SAITS inference successfully imputed {missing_cells_e:,} missing external validation data points.")
else:
    print("Raw eICU tensor file not found. Ensure path is correct.")

 eICU TENSOR SPARSITY (PRE-SAITS IMPUTATION)
Tensor Shape       : (7628, 24, 30) -> (Patients, 24-Hours, Features)
Total Data Cells   : 5,492,160
Missing Cells      : 4,387,102
Sparsity Rate      : 79.88%

-> Locked SAITS inference successfully imputed 4,387,102 missing external validation data points.


In [8]:
print("=========================================================================================")
print(" TRUE TENSOR DENSITY AUDIT (Pre-Imputation Bins) + STATIC FEATURES")
print("=========================================================================================")

# Additional Paths for Feature Names
MIMIC_FEATS = BASE_DIR / "data" / "processed" / "mimiciv" / "mimic_sepsis_tensor_features.npy"
EICU_FEATS = BASE_DIR / "data" / "processed" / "eicu" / "eicu_sepsis_tensor_features.npy"

# Load 3D Tensors: Shape = [Patients, 24 Hours, Features]
m_X = np.load(MIMIC_RAW_TENSOR)
e_X = np.load(EICU_RAW_TENSOR)

m_feats = list(np.load(MIMIC_FEATS))
e_feats = list(np.load(EICU_FEATS))

m_pts, m_steps, m_f_count = m_X.shape
e_pts, e_steps, e_f_count = e_X.shape

m_total_cells = m_pts * m_steps
e_total_cells = e_pts * e_steps

print(f"{'Feature':<15} | {'MIMIC N (Hrs/Pts)':<17} | {'eICU N (Hrs/Pts)':<17} | {'M_Dens%':<7} | {'e_Dens%':<7} | {'Δ Diff%':<8}")
print("-" * 92)

# ==========================================
# 1. STATIC FEATURES (Density per Patient)
# ==========================================
print(" [ STATIC FEATURES (Density per Patient) ]")

# Find CCI and SOFA columns dynamically
cci_col_m = next((col for col in df_mimic.columns if 'charlson' in col.lower() or 'cci' in col.lower()), None)
cci_col_e = next((col for col in df_eicu.columns if 'charlson' in col.lower() or 'cci' in col.lower()), None)

sofa_col_m = next((col for col in df_mimic.columns if 'sofa' in col.lower()), None)
sofa_col_e = next((col for col in df_eicu.columns if 'sofa' in col.lower()), None)

static_map = {
    'AGE': ('age', 'age'),
    'GENDER': ('gender', 'gender'),
    'SOFA': (sofa_col_m, sofa_col_e),
    'CCI': (cci_col_m, cci_col_e)
}

for feat_name, (m_col, e_col) in static_map.items():
    # MIMIC
    if m_col and m_col in df_mimic.columns:
        m_n = df_mimic[m_col].notna().sum()
        m_dens = (m_n / n_mimic) * 100
    else:
        m_n, m_dens = 0, 0.0
        
    # eICU
    if e_col and e_col in df_eicu.columns:
        e_n = df_eicu[e_col].notna().sum()
        e_dens = (e_n / n_eicu) * 100
    else:
        e_n, e_dens = 0, 0.0
        
    d_diff = abs(m_dens - e_dens)
    print(f"{feat_name:<15} | {m_n:<17,} | {e_n:<17,} | {m_dens:>6.2f}% | {e_dens:>6.2f}% | {d_diff:>6.2f}%")

print("-" * 92)

# ==========================================
# 2. TEMPORAL FEATURES (Density per Hour)
# ==========================================
print(" [ TEMPORAL FEATURES (Density per Patient-Hour) ]")

for i, feature in enumerate(m_feats):
    m_slice = m_X[:, :, i]
    e_slice = e_X[:, :, i]
    
    m_n = int(np.sum(~np.isnan(m_slice)))
    e_n = int(np.sum(~np.isnan(e_slice)))
    
    m_dens = (m_n / m_total_cells) * 100
    e_dens = (e_n / e_total_cells) * 100
    
    d_diff = abs(m_dens - e_dens)
    flag = "🚨" if e_dens == 0.0 or d_diff > 30.0 else ""
    
    print(f"{feature.upper():<15} | {m_n:<17,} | {e_n:<17,} | {m_dens:>6.2f}% | {e_dens:>6.2f}% | {d_diff:>6.2f}% {flag}")
    
m_overall = np.mean(~np.isnan(m_X)) * 100
e_overall = np.mean(~np.isnan(e_X)) * 100

print("-" * 92)
print(f"{'OVERALL TENSOR':<15} | {'-':<17} | {'-':<17} | {m_overall:>6.2f}% | {e_overall:>6.2f}% | {abs(m_overall-e_overall):>6.2f}%")
print("=========================================================================================")

 TRUE TENSOR DENSITY AUDIT (Pre-Imputation Bins) + STATIC FEATURES
Feature         | MIMIC N (Hrs/Pts) | eICU N (Hrs/Pts)  | M_Dens% | e_Dens% | Δ Diff% 
--------------------------------------------------------------------------------------------
 [ STATIC FEATURES (Density per Patient) ]
AGE             | 13,015            | 7,628             | 100.00% | 100.00% |   0.00%
GENDER          | 13,015            | 7,628             | 100.00% | 100.00% |   0.00%
SOFA            | 13,011            | 7,583             |  99.97% |  99.41% |   0.56%
CCI             | 13,015            | 7,628             | 100.00% | 100.00% |   0.00%
--------------------------------------------------------------------------------------------
 [ TEMPORAL FEATURES (Density per Patient-Hour) ]
HR              | 293,469           | 178,409           |  93.95% |  97.45% |   3.50% 
MAP             | 281,861           | 173,975           |  90.24% |  95.03% |   4.79% 
RR              | 291,343           | 163,685    

In [11]:
print("==========================================================")
print(" SUPPLEMENTARY TABLE: eICU VASOPRESSOR EXTRACTION AUDIT")
print("==========================================================")

# Paths to the intermediate files
RAW_PRESSORS_FILE = BASE_DIR / "data" / "processed" / "eicu" / "eicu_extracted_pressors_raw.parquet"
STANDARDIZED_PRESSORS_FILE = BASE_DIR / "data" / "processed" / "eicu" / "eicu_standardized_pressors.parquet"
UNPROCESSABLE_FILE = BASE_DIR / "outputs" / "metrics" / "eicu_unprocessable_pressors.csv"

if RAW_PRESSORS_FILE.exists() and STANDARDIZED_PRESSORS_FILE.exists():
    df_raw_pressors = pl.read_parquet(RAW_PRESSORS_FILE).to_pandas()
    df_std_pressors = pl.read_parquet(STANDARDIZED_PRESSORS_FILE).to_pandas()
    
    total_raw = len(df_raw_pressors)
    total_std = len(df_std_pressors)
    
    print(f"Total Regex-Identified Pressor Records : {total_raw:,}")
    print("\n[Raw Unit Heterogeneity]")
    unit_counts = df_raw_pressors['embedded_unit'].value_counts().head(10)
    for unit, count in unit_counts.items():
        print(f"  - {unit:<15}: {count:>8,} ({(count/total_raw*100):>5.1f}%)")
        
    print(f"\n[Data Attrition]")
    if UNPROCESSABLE_FILE.exists():
        df_unproc = pd.read_csv(UNPROCESSABLE_FILE)
        unproc_count = len(df_unproc)
        print(f"  - Unprocessable Records Dropped        : {unproc_count:,} ({(unproc_count/total_raw*100):.1f}%)")
    
    bounds_dropped = total_raw - total_std - (unproc_count if UNPROCESSABLE_FILE.exists() else 0)
    print(f"  - Dropped via Strict Clinical Bounds   : {bounds_dropped:,} ({(bounds_dropped/total_raw*100):.1f}%)")
    print(f"  - Final Standardized Valid Records     : {total_std:,} ({(total_std/total_raw*100):.1f}%)")
else:
    print("Intermediate pressor parquet files not found.")

 SUPPLEMENTARY TABLE: eICU VASOPRESSOR EXTRACTION AUDIT
Total Regex-Identified Pressor Records : 138,536

[Raw Unit Heterogeneity]
  - ml/hr          :   63,459 ( 45.8%)
  - mcg/min        :   42,638 ( 30.8%)
  - mcg/kg/min     :   13,791 ( 10.0%)
  - units/min      :   11,195 (  8.1%)
  - ml             :      630 (  0.5%)
  - units/hr       :      156 (  0.1%)
  - mg/min         :       54 (  0.0%)
  - mg/kg/min      :       31 (  0.0%)
  - mcg/hr         :       28 (  0.0%)
  - mg/hr          :       13 (  0.0%)

[Data Attrition]
  - Unprocessable Records Dropped        : 7,240 (5.2%)
  - Dropped via Strict Clinical Bounds   : 384 (0.3%)
  - Final Standardized Valid Records     : 130,912 (94.5%)


In [10]:
print("==========================================================")
print(" SUPPLEMENTARY TABLE: eICU CONVERSION PATHWAYS & WEIGHTS")
print("==========================================================")

if STANDARDIZED_PRESSORS_FILE.exists():
    print("[Conversion Methodology Applied]")
    pathways = df_std_pressors['conversion_method'].value_counts()
    for method, count in pathways.items():
        print(f"  - {method:<30}: {count:>8,} ({(count/total_std*100):>5.1f}%)")
        
    print("\n[Patient Weight Source for Normalization]")
    weight_sources = df_std_pressors['weight_source'].value_counts()
    for source, count in weight_sources.items():
        print(f"  - {source:<30}: {count:>8,} ({(count/total_std*100):>5.1f}%)")
        
    print("\n* Methodological Note for Reviewers:")
    print("  'concentration_assumed' denotes records recorded in volumetric rates (ml/hr).")
    print("  These were salvaged using standardized critical care infusion concentrations:")
    print("  Norepinephrine (16 mcg/mL), Epinephrine (16 mcg/mL), Phenylephrine (80 mcg/mL),")
    print("  Dopamine (1600 mcg/mL), and Vasopressin (0.2 units/mL).")

 SUPPLEMENTARY TABLE: eICU CONVERSION PATHWAYS & WEIGHTS
[Conversion Methodology Applied]
  - concentration_assumed         :   63,288 ( 48.3%)
  - weight_normalized             :   42,587 ( 32.5%)
  - direct                        :   24,855 ( 19.0%)
  - time_normalized               :      154 (  0.1%)
  - time_and_weight_normalized    :       28 (  0.0%)

[Patient Weight Source for Normalization]
  - measured                      :  129,750 ( 99.1%)
  - imputed_80kg                  :    1,162 (  0.9%)

* Methodological Note for Reviewers:
  'concentration_assumed' denotes records recorded in volumetric rates (ml/hr).
  These were salvaged using standardized critical care infusion concentrations:
  Norepinephrine (16 mcg/mL), Epinephrine (16 mcg/mL), Phenylephrine (80 mcg/mL),
  Dopamine (1600 mcg/mL), and Vasopressin (0.2 units/mL).


In [12]:
import json

print("==========================================================")
print(" SUPPLEMENTARY METRICS: NEQ FEATURE EQUIVALENCE")
print("==========================================================")

NEQ_REPORT_FILE = BASE_DIR / "outputs" / "metrics" / "eicu_feature_equivalence_report_NEQ.json"

if NEQ_REPORT_FILE.exists():
    with open(NEQ_REPORT_FILE, "r") as f:
        neq_report = json.load(f)
        
    print(f"Pharmacological Standard Applied : {neq_report.get('Mathematical_Conversion')}")
    print(f"Total eICU Patients Treated      : {neq_report.get('eICU_Treated_Events'):,}")
    print(f"Max Concurrent Pressors (eICU)   : {neq_report.get('eICU_Max_Concurrent_Pressors')}")
    print("-" * 58)
    print(f"MIMIC-IV Median NEQ (Hourly Max) : {neq_report.get('MIMIC_Median_NEQ_Hourly_Max'):.3f} mcg/kg/min")
    print(f"eICU Median NEQ (Hourly Max)     : {neq_report.get('eICU_Median_NEQ_Hourly_Max'):.3f} mcg/kg/min")
    print("-" * 58)
    print("Conclusion: The extracted and standardized eICU NEQ distribution demonstrates")
    print("a lower median intervention dose compared to the MIMIC-IV cohort. This reflects")
    print("expected epidemiological heterogeneity: MIMIC represents a single quaternary academic")
    print("center (higher baseline severity), whereas eICU comprises diverse community hospitals.")
    print("This intrinsic physiological domain shift structurally explains the necessity of")
    print("external probability recalibration during model deployment.")
else:
    print("NEQ feature equivalence report not found.")

 SUPPLEMENTARY METRICS: NEQ FEATURE EQUIVALENCE
Pharmacological Standard Applied : Brown et al. (2013) Pure Implementation
Total eICU Patients Treated      : 102,629
Max Concurrent Pressors (eICU)   : 5
----------------------------------------------------------
MIMIC-IV Median NEQ (Hourly Max) : 0.200 mcg/kg/min
eICU Median NEQ (Hourly Max)     : 0.103 mcg/kg/min
----------------------------------------------------------
Conclusion: The extracted and standardized eICU NEQ distribution demonstrates
a lower median intervention dose compared to the MIMIC-IV cohort. This reflects
expected epidemiological heterogeneity: MIMIC represents a single quaternary academic
center (higher baseline severity), whereas eICU comprises diverse community hospitals.
This intrinsic physiological domain shift structurally explains the necessity of
external probability recalibration during model deployment.


In [13]:
from sklearn.metrics import roc_auc_score

print("==========================================================")
print(" RECONSTRUCTING 122D FEATURE SPACE FOR PORTABILITY MAP")
print("==========================================================")

# [FIXED P4-2]
# This cell used to hstack the full 8-column static array, producing a 128-D
# space that was mislabelled 124-D. Four of those columns (race, admission_type,
# first_careunit, baseline_pf_ratio) are not model inputs, and the first three
# are label-encoded nominal codes assigned independently in each database, so a
# univariate AUROC over them was meaningless and not comparable across cohorts.
# We now use exactly the four static variables the model consumes, in the model's
# order, so feature_names matches mimic_champion_features.json one for one.
MODEL_STATIC_ORDER = ["age", "baseline_sofa"]

# Load MIMIC Arrays (allow_pickle=True for static mixed-type arrays)
m_X_imp = np.load(BASE_DIR / "data" / "processed" / "mimiciv" / "mimic_sepsis_imputed_tensor.npy")
m_X_stat = np.load(BASE_DIR / "data" / "processed" / "mimiciv" / "mimic_sepsis_tensor_static.npy", allow_pickle=True)
m_y = np.load(BASE_DIR / "data" / "processed" / "mimiciv" / "mimic_sepsis_tensor_labels.npy")

m_temp_feats = np.load(BASE_DIR / "data" / "processed" / "mimiciv" / "mimic_sepsis_tensor_features.npy", allow_pickle=True)
m_stat_feats = list(np.load(BASE_DIR / "data" / "processed" / "mimiciv" / "mimic_sepsis_tensor_static_features.npy", allow_pickle=True))

# Load eICU Arrays
e_X_imp = np.load(BASE_DIR / "data" / "processed" / "eicu" / "eicu_sepsis_imputed_tensor.npy")
e_X_stat = np.load(BASE_DIR / "data" / "processed" / "eicu" / "eicu_sepsis_tensor_static.npy", allow_pickle=True)
e_y = np.load(BASE_DIR / "data" / "processed" / "eicu" / "eicu_sepsis_tensor_labels.npy")

e_stat_feats = list(np.load(BASE_DIR / "data" / "processed" / "eicu" / "eicu_sepsis_tensor_static_features.npy", allow_pickle=True))
assert m_stat_feats == e_stat_feats, "Static feature order differs between cohorts"

# Column positions of the four model statics inside the saved 8-column array
stat_idx = [m_stat_feats.index(c) for c in MODEL_STATIC_ORDER]
print(f"Static columns kept : {MODEL_STATIC_ORDER}")
print(f"Their positions     : {stat_idx}  (of {len(m_stat_feats)} saved)")
print(f"Dropped as non-model: {[c for c in m_stat_feats if c not in MODEL_STATIC_ORDER]}")

# Flattening Function (Mean, Min, Max, Std)
def flatten_tensor(X_temp, X_stat):
    X_mean = np.mean(X_temp, axis=1)
    X_min = np.min(X_temp, axis=1)
    X_max = np.max(X_temp, axis=1)
    X_std = np.std(X_temp, axis=1)

    # Model order is [4 static, 120 temporal]
    flat = np.hstack([X_stat[:, stat_idx], X_mean, X_min, X_max, X_std])

    # Handle pandas/numpy object artifacts (e.g., ">89" to 89.0) safely
    df_flat = pd.DataFrame(flat)
    df_flat = df_flat.apply(pd.to_numeric, errors="coerce").fillna(0)
    return df_flat.values.astype(np.float32)

m_flat = flatten_tensor(m_X_imp, m_X_stat)
e_flat = flatten_tensor(e_X_imp, e_X_stat)

# Feature names, identical in both cohorts and matching the champion feature list
feature_names = list(MODEL_STATIC_ORDER)
for stat in ["Mean", "Min", "Max", "Std"]:
    for f in m_temp_feats:
        feature_names.append(f"{f}_{stat}")

assert m_flat.shape[1] == len(feature_names) == 122, (
    f"expected 122 columns, got {m_flat.shape[1]} / {len(feature_names)}"
)

# Cross-check against the list the locked model was trained on
champ_file = BASE_DIR / "outputs" / "features" / "mimic_champion_features.json"
if champ_file.exists():
    with open(champ_file) as f:
        champ_feats = json.load(f)
    if champ_feats == feature_names:
        print("Feature order matches mimic_champion_features.json exactly.")
    else:
        print("[WARN] Feature order differs from mimic_champion_features.json!")

print(f"MIMIC 122D Shape : {m_flat.shape} | Labels: {m_y.shape}")
print(f"eICU 122D Shape  : {e_flat.shape} | Labels: {e_y.shape}")
print("Feature space successfully reconstructed.")


 RECONSTRUCTING 124D FEATURE SPACE FOR PORTABILITY MAP
MIMIC 124D Shape : (13015, 128) | Labels: (13015,)
eICU 124D Shape  : (7628, 128) | Labels: (7628,)
Feature space successfully reconstructed.


In [14]:
print("==========================================================")
print(" COMPUTING PROGNOSTIC PORTABILITY RATIO (R_j)")
print("==========================================================")

def get_directional_auc(y_true, y_score):
    """Calculates AUROC and flips it if it's < 0.5 to measure raw discriminative magnitude."""
    # Add minor noise to avoid constant-value errors
    y_score = y_score + np.random.normal(0, 1e-9, size=len(y_score))
    auc = roc_auc_score(y_true, y_score)
    return auc if auc >= 0.5 else 1.0 - auc

portability_results = []

for i, feat in enumerate(feature_names):
    auc_m = get_directional_auc(m_y, m_flat[:, i])
    auc_e = get_directional_auc(e_y, e_flat[:, i])
    
    # Discriminative Signal Above Chance (P - 0.5)
    p_m = auc_m - 0.5
    p_e = auc_e - 0.5
    
    # Only calculate R_j for features that actually have meaningful prognostic 
    # signal in the source dataset (AUROC > 0.55) to avoid dividing by noise.
    if auc_m > 0.55:
        r_j = p_e / p_m
        
        # Categorize Portability
        if r_j >= 0.8:
            status = "🟢 High"
        elif r_j >= 0.4:
            status = "🟡 Partial"
        else:
            status = "🔴 Poor"
            
        portability_results.append({
            "Feature": feat,
            "MIMIC_AUROC": auc_m,
            "eICU_AUROC": auc_e,
            "R_j": r_j,
            "Status": status
        })

df_portability = pd.DataFrame(portability_results).sort_values("MIMIC_AUROC", ascending=False)
print(f"Computed portability for {len(df_portability)} prognostically significant features.")

 COMPUTING PROGNOSTIC PORTABILITY RATIO (R_j)
Computed portability for 95 prognostically significant features.


In [17]:
print("=====================================================================================")
print(" PROGNOSTIC PORTABILITY MAP (Top 30 Source Features)")
print("=====================================================================================")
print(f"{'Feature':<25} | {'MIMIC AUROC':<12} | {'eICU AUROC':<12} | {'R_j (Retained)':<15} | {'Status'}")
print("-" * 85)

for _, row in df_portability.head(30).iterrows():
    feat = row['Feature']
    m_auc = row['MIMIC_AUROC']
    e_auc = row['eICU_AUROC']
    r_j = row['R_j'] * 100
    status = row['Status']
    
    print(f"{feat:<25} | {m_auc:<12.3f} | {e_auc:<12.3f} | {r_j:>6.1f}%          | {status}")

print("-" * 85)
print("\n[Portability Summary across all significant features]")
print(df_portability['Status'].value_counts().to_string())
print("\nInterpretation for Manuscript:")
print("R_j quantifies the fraction of above-chance discrimination retained externally.")
print("This uncovers exactly which biological domains degrade under EHR domain shift.")

 PROGNOSTIC PORTABILITY MAP (Top 30 Source Features)
Feature                   | MIMIC AUROC  | eICU AUROC   | R_j (Retained)  | Status
-------------------------------------------------------------------------------------
lactate_Mean              | 0.749        | 0.719        |   88.1%          | 🟢 High
aptt_Mean                 | 0.737        | 0.711        |   89.0%          | 🟢 High
pt_Mean                   | 0.734        | 0.703        |   86.7%          | 🟢 High
creatinine_Mean           | 0.731        | 0.650        |   65.0%          | 🟡 Partial
creatinine_Max            | 0.724        | 0.626        |   56.2%          | 🟡 Partial
lactate_Max               | 0.722        | 0.708        |   93.7%          | 🟢 High
pt_Max                    | 0.712        | 0.685        |   87.1%          | 🟢 High
bun_Mean                  | 0.708        | 0.617        |   55.9%          | 🟡 Partial
albumin_Mean              | 0.708        | 0.726        |  108.6%          | 🟢 High
aptt_Max     

In [15]:
# [NOTE P4-2 / P1-6]
# eicu_ot_pruned_tensor.npy is written by 04_atlas_datasets/05 AFTER the MIMIC
# training scalers are applied, whereas m_flat_75 / e_flat_75 below are in raw
# clinical units. That difference does not affect this comparison: StandardScaler
# is a monotone per-column transform, and AUROC depends only on ranking, so R_j
# is unchanged. Do not "fix" it by rescaling one side.

print("==========================================================")
print(" RECONSTRUCTING PRUNED FEATURE SPACE FOR PORTABILITY")
print("==========================================================")

# Load the exact 75 feature names
with open(BASE_DIR / "outputs" / "features" / "mimic_stable_optimal_features.json", "r") as f:
    pruned_feats = json.load(f)

# m_flat and e_flat are the 124-D model space built above.
# Slice them to the RFECV stable subset; the size is whatever the JSON holds.
feat_indices_pruned = [feature_names.index(feat) for feat in pruned_feats]

m_flat_75 = m_flat[:, feat_indices_pruned]
e_flat_75 = e_flat[:, feat_indices_pruned]

# Load the OT-adapted Pruned tensor we just saved
e_ot_flat_75 = np.load(BASE_DIR / "outputs" / "features" / "eicu_ot_pruned_tensor.npy")

print(f"MIMIC 75D Shape     : {m_flat_75.shape}")
print(f"eICU (Raw) 75D      : {e_flat_75.shape}")
print(f"eICU (OT) 75D       : {e_ot_flat_75.shape}")
print("Pruned feature space successfully reconstructed.")

 RECONSTRUCTING 75D PRUNED FEATURE SPACE FOR PORTABILITY
MIMIC 75D Shape     : (13015, 75)
eICU (Raw) 75D      : (7628, 75)
eICU (OT) 75D       : (7628, 75)
Pruned feature space successfully reconstructed.


In [16]:
print("=========================================================================================")
print(" PRUNED PORTABILITY MAP: RAW eICU vs. OT-HARMONIZED eICU")
print("=========================================================================================")

pruned_portability = []

for i, feat in enumerate(pruned_feats):
    # Base AUROCs
    auc_m = get_directional_auc(m_y, m_flat_75[:, i])
    auc_e_raw = get_directional_auc(e_y, e_flat_75[:, i])
    auc_e_ot = get_directional_auc(e_y, e_ot_flat_75[:, i])
    
    # Discriminative Signal Above Chance (P - 0.5)
    p_m = auc_m - 0.5
    p_e_raw = auc_e_raw - 0.5
    p_e_ot = auc_e_ot - 0.5
    
    if auc_m > 0.55:
        # Portability Ratios
        rj_raw = (p_e_raw / p_m) * 100
        rj_ot = (p_e_ot / p_m) * 100
        
        # Calculate how much OT changed the portability
        ot_shift = rj_ot - rj_raw
        
        pruned_portability.append({
            "Feature": feat,
            "MIMIC_AUC": auc_m,
            "Raw_Rj": rj_raw,
            "OT_Rj": rj_ot,
            "OT_Impact": ot_shift
        })

df_pruned_port = pd.DataFrame(pruned_portability).sort_values("MIMIC_AUC", ascending=False)

print(f"{'Feature':<25} | {'MIMIC AUC':<10} | {'Raw R_j':<10} | {'OT R_j':<10} | {'OT Impact'}")
print("-" * 75)

for _, row in df_pruned_port.head(124).iterrows():
    feat = row['Feature'][:25]
    m_auc = row['MIMIC_AUC']
    raw_rj = row['Raw_Rj']
    ot_rj = row['OT_Rj']
    impact = row['OT_Impact']
    
    # Visual flag for destructive OT
    flag = "⚠️ Destructive" if impact < -10 else ("✅ Restorative" if impact > 10 else "Neutral")
    
    print(f"{feat:<25} | {m_auc:<10.3f} | {raw_rj:>7.1f}%   | {ot_rj:>7.1f}%   | {impact:>6.1f}% ({flag})")

print("-" * 75)

 PRUNED PORTABILITY MAP: RAW eICU vs. OT-HARMONIZED eICU
Feature                   | MIMIC AUC  | Raw R_j    | OT R_j     | OT Impact
---------------------------------------------------------------------------
lactate_Mean              | 0.749      |    88.1%   |    48.3%   |  -39.8% (⚠️ Destructive)
aptt_Mean                 | 0.737      |    89.0%   |    42.6%   |  -46.5% (⚠️ Destructive)
pt_Mean                   | 0.734      |    86.7%   |    37.7%   |  -49.0% (⚠️ Destructive)
creatinine_Mean           | 0.731      |    65.0%   |    95.6%   |   30.6% (✅ Restorative)
creatinine_Max            | 0.724      |    56.2%   |    97.6%   |   41.4% (✅ Restorative)
pt_Max                    | 0.712      |    87.1%   |    15.2%   |  -71.9% (⚠️ Destructive)
bun_Mean                  | 0.708      |    55.9%   |    51.1%   |   -4.8% (Neutral)
albumin_Mean              | 0.708      |   108.6%   |    14.0%   |  -94.6% (⚠️ Destructive)
aptt_Max                  | 0.701      |    99.7%   |    90.9% 